<center><h1>Character-level representation vs. word embeddings for sentiment classification</h1></center>

Character-level features can be very helpful because they tend to be more robust to typos, slang, and
out-of-vocabulary words — all of which are common in app reviews ("gud", "veryyy bad", "crashess", ...).
In this notebook we compare a **character n-gram TF-IDF representation** with the **spaCy word-embedding
representation**, keeping the model (a `tf.keras` neural network) and the preprocessing identical.

In [2]:
!python -m spacy download en_core_web_md

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 33.5/33.5 MB 29.4 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_md')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [3]:
import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'

import spacy
import pandas as pd
import numpy as np
import re
import tensorflow as tf

nlp = spacy.load('en_core_web_md')
tf.random.set_seed(42)

In [4]:
from google.colab import files
uploaded = files.upload()
dataset = pd.read_csv('all_data.csv')
dataset = dataset.dropna(subset=['review', 'sentiment']).reset_index(drop=True)
dataset.head()

Saving all_data.csv to all_data.csv


,review,sentiment
0,Aditya Ingole Deaf,2
1,I love the app.! There is no issue but if u co...,1
2,"So hard to use. The web app failed, and the mo...",0
3,I hate that the app makes a sound every time s...,1
4,Useless at BSE star MF meet.voice too mych slo...,0


In [5]:
def preprocess_text(text):
    text = str(text).lower()
    text = re.sub(r'http\S+|www\.\S+', ' ', text)
    text = re.sub(r'[^a-zA-Z\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

dataset['clean_text'] = dataset['review'].map(preprocess_text)
dataset = dataset[dataset['clean_text'].str.len() > 0].reset_index(drop=True)
dataset.shape

(39804, 3)

In [6]:
def get_longest_sequence(texts):
    return max(len(t) for t in texts)

longest_seq = get_longest_sequence(dataset['clean_text'])
print(f"Longest review (characters): {longest_seq}")

Longest review (characters): 2453


There are some reviews written with non-English characters — since we're using an English spaCy model, our cleaning step already strips those out, so both representations see the exact same character/word set.

In [7]:
from collections import Counter

char_counts = Counter("".join(dataset['clean_text'].tolist()))
char_counts.most_common(10)

[(' ', 694106),
 ('e', 362363),
 ('t', 284270),
 ('o', 263204),
 ('i', 242736),
 ('a', 235028),
 ('n', 222537),
 ('s', 204565),
 ('r', 146921),
 ('l', 122888)]

In [8]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    dataset['clean_text'], dataset['sentiment'], test_size=0.2, random_state=42, stratify=dataset['sentiment']
)
y_train_arr = np.array(y_train)
y_test_arr = np.array(y_test)
len(X_train), len(X_test)

(31843, 7961)

Now prepare the character-level representation

In [9]:
from sklearn.feature_extraction.text import TfidfVectorizer

char_vectorizer = TfidfVectorizer(analyzer='char', ngram_range=(2, 4), max_features=5000)
X_train_char = char_vectorizer.fit_transform(X_train).toarray().astype('float32')
X_test_char = char_vectorizer.transform(X_test).toarray().astype('float32')
X_train_char.shape

(31843, 5000)

In [10]:
char_vectorizer.get_feature_names_out()[:20]

array([' a', ' a ', ' a b', ' a c', ' a d', ' a f', ' a g', ' a h',
       ' a l', ' a m', ' a n', ' a p', ' a r', ' a s', ' a t', ' a v',
       ' a w', ' aa', ' ab', ' abl'], dtype=object)

now let's prepare the word-embedding representation

In [11]:
def encode_texts(texts, nlp_model):
    return np.array([doc.vector for doc in nlp_model.pipe(texts, batch_size=256)], dtype='float32')

X_train_emb = encode_texts(X_train, nlp)
X_test_emb = encode_texts(X_test, nlp)
X_train_emb.shape

(31843, 300)

now let's build the model

In [12]:
def build_model(input_dim, num_classes=3):
    model = tf.keras.models.Sequential([
        tf.keras.layers.Input(shape=(input_dim,)),
        tf.keras.layers.Dense(128, activation='relu'),
        tf.keras.layers.Dropout(0.3),
        tf.keras.layers.Dense(64, activation='relu'),
        tf.keras.layers.Dense(num_classes, activation='softmax')
    ])
    model.compile(optimizer='adam',
                  loss='sparse_categorical_crossentropy',
                  metrics=['accuracy'])
    return model

In [13]:
model_char = build_model(input_dim=X_train_char.shape[1])
history_char = model_char.fit(X_train_char, y_train_arr, epochs=8, batch_size=128,
                               validation_split=0.1, verbose=0)
print("Final training accuracy:", history_char.history['accuracy'][-1])

Final training accuracy: 0.8667736649513245


In [14]:
model_emb = build_model(input_dim=X_train_emb.shape[1])
history_emb = model_emb.fit(X_train_emb, y_train_arr, epochs=8, batch_size=128,
                             validation_split=0.1, verbose=0)
print("Final training accuracy:", history_emb.history['accuracy'][-1])

Final training accuracy: 0.568183422088623


## Evaluate

In [15]:
from sklearn.metrics import f1_score, classification_report

test_loss_char, test_acc_char = model_char.evaluate(X_test_char, y_test_arr, verbose=0)
pred_char = np.argmax(model_char.predict(X_test_char, verbose=0), axis=1)
f1_char = f1_score(y_test_arr, pred_char, average='macro')
print(f"Char n-grams (TF-IDF) -> accuracy: {test_acc_char:.4f}, macro F1: {f1_char:.4f}")
print(classification_report(y_test_arr, pred_char))

Char n-grams (TF-IDF) -> accuracy: 0.7017, macro F1: 0.6867
              precision    recall  f1-score   support

           0       0.74      0.74      0.74      2657
           1       0.62      0.50      0.56      2302
           2       0.71      0.82      0.76      3002

    accuracy                           0.70      7961
   macro avg       0.69      0.69      0.69      7961
weighted avg       0.70      0.70      0.70      7961



In [16]:
test_loss_emb, test_acc_emb = model_emb.evaluate(X_test_emb, y_test_arr, verbose=0)
pred_emb = np.argmax(model_emb.predict(X_test_emb, verbose=0), axis=1)
f1_emb = f1_score(y_test_arr, pred_emb, average='macro')
print(f"spaCy embeddings -> accuracy: {test_acc_emb:.4f}, macro F1: {f1_emb:.4f}")
print(classification_report(y_test_arr, pred_emb))

spaCy embeddings -> accuracy: 0.5635, macro F1: 0.5287
              precision    recall  f1-score   support

           0       0.55      0.67      0.61      2657
           1       0.46      0.25      0.32      2302
           2       0.61      0.71      0.66      3002

    accuracy                           0.56      7961
   macro avg       0.54      0.54      0.53      7961
weighted avg       0.55      0.56      0.54      7961



In [17]:
def predict_char(text):
    clean = preprocess_text(text)
    vec = char_vectorizer.transform([clean]).toarray().astype('float32')
    pred = np.argmax(model_char.predict(vec, verbose=0), axis=1)[0]
    return {0: 'negative', 1: 'neutral', 2: 'positive'}[pred]

predict_char("app crashess every time i try to opne it")

'negative'

## Conclusion

- The character n-gram representation was able to handle the typo above ("crashess", "opne") reasonably well,
  since it doesn't rely on exact-word matches the way word-level representations do.
- Both representations were fed into the exact same `tf.keras` neural network architecture, so the difference
  in scores reflects a real difference between representations, not between models.
- Same classifier, same preprocessing pipeline for both — only the way the text is vectorized changed.